<a href="https://colab.research.google.com/github/ashutosh830/CEI_Program_Assignment/blob/main/week8_Ashutosh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

##tool calculator

In [2]:
def calculator(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

##tool keyword extraction

In [3]:
def extract_keywords(text: str) -> list:
    try:
        words = text.split()
        keywords = list(
            set(
                [w.lower() for w in words if len(w) > 4]
            )
        )
        return keywords[:5]
    except Exception:
        return []

## define state of graph

In [6]:
class AgentState(TypedDict):
    query: str
    route: str
    response: dict

##router node

In [7]:
def router_node(state: AgentState):

    query = state["query"].lower()

    if "calculate" in query:
        state["route"] = "calculator"

    elif "keywords" in query:
        state["route"] = "keywords"

    else:
        state["route"] = "general"

    return state

##node1 function - calculator

In [8]:
def calculator_node(state: AgentState):

    query = state["query"]

    expression = query.lower().replace("calculate", "").strip()

    result = calculator(expression)

    state["response"] = {
        "type": "calculation",
        "result": result
    }

    return state

##node2 keyword node

In [9]:
def keyword_node(state: AgentState):

    query = state["query"]

    text = query.lower().replace(
        "extract keywords from",
        ""
    ).strip()

    result = extract_keywords(text)

    state["response"] = {
        "type": "keywords",
        "result": result
    }

    return state

##node4 general node

In [10]:
def general_node(state: AgentState):

    state["response"] = {
        "type": "general",
        "result": "This is a general response."
    }

    return state

##route the nodes

In [11]:
def route_decision(state: AgentState):
    return state["route"]

##using langgraph framework

In [12]:
builder = StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("calculator", calculator_node)
builder.add_node("keywords", keyword_node)
builder.add_node("general", general_node)

builder.set_entry_point("router")

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "calculator": "calculator",
        "keywords": "keywords",
        "general": "general"
    }
)

builder.add_edge("calculator", END)
builder.add_edge("keywords", END)
builder.add_edge("general", END)

graph = builder.compile()

##functioning of agents

In [13]:
def agent(query: str):

    try:

        result = graph.invoke(
            {
                "query": query,
                "route": "",
                "response": {}
            }
        )

        return result["response"]

    except Exception as e:

        return {
            "type": "error",
            "result": str(e)
        }

##test the agents

In [14]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['industries', 'intelligence', 'transforming', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'This is a general response.'}
--------------------------------------------------


##interaction

In [16]:
while True:

    user_input = input(
        "\nEnter query (type 'exit' to stop): "
    )

    if user_input.lower() == "exit":
        break

    print("Response:", agent(user_input))


Enter query (type 'exit' to stop): calculate 20+3
Response: {'type': 'calculation', 'result': '23'}

Enter query (type 'exit' to stop): what is machine learning
Response: {'type': 'general', 'result': 'This is a general response.'}

Enter query (type 'exit' to stop): exit
